# 00 · Prohori end to end (Kaggle / Colab / laptop)

Runs the full pipeline: **generate → validate (T1–T14) → features → train → Agent Watch → evaluate → export**.

| Step | Time on a laptop / Kaggle CPU (measured, full scale) |
|---|---|
| Generate ~680k transactions | ~10 s |
| Validate T1–T14 | ~25 s |
| Streaming features + hourly graph | ~1 min |
| Train (Optuna 30 trials) | ~8–15 min (set `TRIALS = 0` for ~1 min) |
| Agent Watch + evaluate (with ablation & SHAP) | ~2 min |

No GPU needed. On Kaggle: add this repo as a Dataset (or set `REPO_URL` in the next cell) and run all.
To keep the generated data, save `/kaggle/working/prohori/data/generated` as a private Kaggle Dataset.

In [ ]:
# --- setup: find the repo root (works locally, in Colab and on Kaggle) -------------------------
import os, sys, shutil, subprocess
from pathlib import Path

REPO_URL = ""          # optional: your GitHub repo URL, cloned if the code is not found
ROOT = None
for cand in [Path.cwd(), Path.cwd().parent, *Path("/kaggle/input").glob("*")] if Path("/kaggle/input").exists() else [Path.cwd(), Path.cwd().parent]:
    if (cand / "src" / "pipeline.py").exists():
        ROOT = cand
        break
if ROOT is None and REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "prohori"], check=True)
    ROOT = Path("prohori").resolve()
assert ROOT is not None, "Put this notebook next to the repo (or add the repo as a Kaggle dataset / set REPO_URL)"
if str(ROOT).startswith("/kaggle/input"):          # Kaggle inputs are read-only: work on a copy
    dst = Path("/kaggle/working/prohori")
    shutil.copytree(ROOT, dst, dirs_exist_ok=True)
    ROOT = dst
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
# Kaggle images already ship most of these; this only fills gaps.
import importlib.util, subprocess, sys
need = [p for p in ("lightgbm", "xgboost", "shap", "optuna", "networkx", "yaml", "pyarrow") if importlib.util.find_spec(p) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[{"yaml": "pyyaml"}.get(p, p) for p in need]], check=True)
print("missing before install:", need or "none")

## Parameters

In [ ]:
SCALE = 1.0      # 1.0 = 10k customers / ~680k txns; 0.1 = quick smoke run
TRIALS = 30      # Optuna trials for LightGBM (0 = use config params)
OUT_ROOT = None  # e.g. "runs/small" to keep a scaled run separate from the full one

In [ ]:
import subprocess, sys
cmd = [sys.executable, "-m", "src.pipeline", "--trials", str(TRIALS)]
if SCALE != 1.0:
    cmd += ["--scale", str(SCALE)]
if OUT_ROOT:
    cmd += ["--out-root", OUT_ROOT]
print(" ".join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
print(r.stdout[-6000:])
print(r.stderr[-3000:])
assert r.returncode == 0, "pipeline failed - see output above"

## Results

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40, "display.width", 200)
from src.common.config import load_config, resolve
from src.models import plots
plots.style()
cfg = load_config(scale=SCALE if SCALE != 1.0 else None)
if OUT_ROOT:
    cfg["paths"]["reports_dir"] = OUT_ROOT + "/reports"
REPORTS = resolve(cfg, "reports_dir")

In [ ]:
display(Markdown((REPORTS / "data_validation.md").read_text(encoding="utf-8")))
m = json.loads((REPORTS / "metrics_test.json").read_text(encoding="utf-8"))
display(pd.DataFrame(m["comparison"]))
display(pd.DataFrame(m["per_scenario"]))
print(json.dumps(m["impact"], indent=1))

In [ ]:
for f in ("pr_curves_test.png", "recall_by_scenario_test.png", "shap_global_top15.png", "agent_watch_sc06.png"):
    p = REPORTS / "figures" / f
    if p.exists():
        display(Image(filename=str(p)))
display(Markdown((REPORTS / "demo_scenarios.md").read_text(encoding="utf-8")))